# AI Programming — Lecture 19
## Further Studies 3: Channel-Independent Patch Transformer + RevIN

Further Studies 2의 Channel-Independent Patch Transformer에
**RevIN (Reversible Instance Normalization)**을 추가합니다.

### 핵심 아이디어
각 sample과 channel의 input window를 개별적으로 normalization합니다.

```text
Input window
→ RevIN normalization
→ Patch Transformer
→ residual forecast
→ RevIN inverse transform
→ final forecast
```

Global StandardScaler와 달리 RevIN은 **각 window의 local mean/std**를 사용합니다.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
import keras
from keras import layers

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

SEED = 42
keras.utils.set_random_seed(SEED)

gpus = tf.config.list_physical_devices("GPU")
for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError:
        pass

print("Python:", sys.version.split()[0])
print("TensorFlow:", tf.__version__)
print("Keras:", keras.__version__)
print("GPU:", gpus)


## 1. ETTh1 데이터 불러오기

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_PATH = Path('/content/drive/MyDrive/Colab Notebooks/data/ETTh1.csv')

df = pd.read_csv(DATA_PATH)

CHANNELS = [
    'HUFL', 'HULL', 'MUFL', 'MULL',
    'LUFL', 'LULL', 'OT'
]

values = df[CHANNELS].values.astype('float32')

print("Data path:", DATA_PATH)
print("Shape:", df.shape)
print("Channels:", CHANNELS)
print(df.head())


## 2. Chronological Split

In [ ]:
LOOKBACK = 96
PRED_LEN = 24
NUM_CHANNELS = len(CHANNELS)

n = len(values)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

train_raw = values[:train_end]
val_raw = values[train_end:val_end]
test_raw = values[val_end:]

print("Train:", train_raw.shape)
print("Validation:", val_raw.shape)
print("Test:", test_raw.shape)

# This scaler is used ONLY to report normalized metrics
# comparable with FS1 and FS2.
report_scaler = StandardScaler()
report_scaler.fit(train_raw)


## 3. Multivariate Forecasting Window 생성

In [ ]:
def create_windows(values, lookback=96, pred_len=24):
    X, Y = [], []
    total_len = lookback + pred_len

    for i in range(len(values) - total_len + 1):
        window = values[i:i + total_len]

        past = window[:lookback]
        future = window[lookback:]

        X.append(past)
        Y.append(future)

    return (
        np.array(X, dtype='float32'),
        np.array(Y, dtype='float32')
    )

X_train_raw, y_train_raw = create_windows(
    train_raw, LOOKBACK, PRED_LEN
)
X_val_raw, y_val_raw = create_windows(
    val_raw, LOOKBACK, PRED_LEN
)
X_test_raw, y_test_raw = create_windows(
    test_raw, LOOKBACK, PRED_LEN
)

print("X_train:", X_train_raw.shape)
print("y_train:", y_train_raw.shape)
print("X_val  :", X_val_raw.shape)
print("X_test :", X_test_raw.shape)


## 4. RevIN — 각 Input Window를 개별 Normalization

각 `(sample, channel)` window에서 mean과 std를 계산합니다.

Forecast를 원래 scale로 복원할 수 있도록
해당 통계량을 함께 저장합니다.

In [ ]:
REVIN_EPS = 1e-5

def apply_revin(X_raw, y_raw, eps=1e-5):
    # Statistics are computed from the input window only.
    # Shapes: [B, 1, C]
    mean = np.mean(
        X_raw,
        axis=1,
        keepdims=True
    )

    var = np.var(
        X_raw,
        axis=1,
        keepdims=True
    )

    std = np.sqrt(var + eps)

    X_norm = (X_raw - mean) / std
    y_norm = (y_raw - mean) / std

    # Residual target in the RevIN-normalized space.
    baseline_norm = X_norm[:, -1:, :]
    y_res_norm = y_norm - baseline_norm

    return (
        X_norm.astype('float32'),
        y_norm.astype('float32'),
        y_res_norm.astype('float32'),
        mean.astype('float32'),
        std.astype('float32')
    )

(
    X_train,
    y_train,
    y_train_res,
    train_mean,
    train_std
) = apply_revin(
    X_train_raw,
    y_train_raw,
    REVIN_EPS
)

(
    X_val,
    y_val,
    y_val_res,
    val_mean,
    val_std
) = apply_revin(
    X_val_raw,
    y_val_raw,
    REVIN_EPS
)

(
    X_test,
    y_test,
    y_test_res,
    test_mean,
    test_std
) = apply_revin(
    X_test_raw,
    y_test_raw,
    REVIN_EPS
)

print("RevIN input shape:", X_train.shape)
print("RevIN mean shape :", train_mean.shape)
print("RevIN std shape  :", train_std.shape)


### RevIN과 Global Standardization의 차이

```text
Global Standardization
→ training dataset 전체의 channel별 mean/std

RevIN
→ 현재 sample/window의 channel별 mean/std
```

RevIN은 시계열의 local distribution shift를 완화하는 데 사용될 수 있습니다.

## 5. Channel-Independent Overlapping Patches

In [ ]:
PATCH_LEN = 16
STRIDE = 8

N_PATCHES = (LOOKBACK - PATCH_LEN) // STRIDE + 1

def create_multivariate_patches(X, patch_len=16, stride=8):
    # X: [B, T, C]
    channel_patches = []

    for c in range(X.shape[2]):
        patches = []

        for start in range(
            0,
            LOOKBACK - patch_len + 1,
            stride
        ):
            patch = X[:, start:start + patch_len, c]
            patches.append(patch)

        # [B, N_PATCHES, PATCH_LEN]
        patches = np.stack(
            patches,
            axis=1
        )
        channel_patches.append(patches)

    # [B, C, N_PATCHES, PATCH_LEN]
    return np.stack(
        channel_patches,
        axis=1
    ).astype('float32')

X_train_patch = create_multivariate_patches(
    X_train, PATCH_LEN, STRIDE
)
X_val_patch = create_multivariate_patches(
    X_val, PATCH_LEN, STRIDE
)
X_test_patch = create_multivariate_patches(
    X_test, PATCH_LEN, STRIDE
)

print("Number of patches per channel:", N_PATCHES)
print("Patched train shape:", X_train_patch.shape)


## 6. Channel을 Independent Sequence로 변환

In [ ]:
def to_channel_independent_inputs(X_patch, y_res):
    B = X_patch.shape[0]

    # [B, C, N, P] -> [B*C, N, P]
    X_ci = X_patch.reshape(
        B * NUM_CHANNELS,
        N_PATCHES,
        PATCH_LEN
    )

    # [B, H, C] -> [B, C, H] -> [B*C, H]
    y_ci = np.transpose(
        y_res,
        (0, 2, 1)
    ).reshape(
        B * NUM_CHANNELS,
        PRED_LEN
    )

    return (
        X_ci.astype('float32'),
        y_ci.astype('float32')
    )

X_train_ci, y_train_ci = to_channel_independent_inputs(
    X_train_patch,
    y_train_res
)
X_val_ci, y_val_ci = to_channel_independent_inputs(
    X_val_patch,
    y_val_res
)
X_test_ci, y_test_ci = to_channel_independent_inputs(
    X_test_patch,
    y_test_res
)

print("Channel-independent train X:", X_train_ci.shape)
print("Channel-independent train y:", y_train_ci.shape)


## 7. Learned Positional Embedding

In [ ]:
class LearnedPositionalEmbedding(layers.Layer):
    def __init__(self, max_len, embed_dim):
        super().__init__()

        self.position_embedding = layers.Embedding(
            input_dim=max_len,
            output_dim=embed_dim
        )

    def call(self, inputs):
        positions = keras.ops.arange(
            0,
            keras.ops.shape(inputs)[1],
            1
        )

        return (
            inputs
            + self.position_embedding(positions)
        )


## 8. Transformer Encoder Block

In [ ]:
class TransformerEncoder(layers.Layer):
    def __init__(
        self,
        embed_dim,
        num_heads,
        ff_dim,
        dropout=0.1
    ):
        super().__init__()

        self.attention = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim // num_heads,
            dropout=dropout
        )

        self.dense1 = layers.Dense(
            ff_dim,
            activation='relu'
        )
        self.dense2 = layers.Dense(embed_dim)

        self.norm1 = layers.LayerNormalization()
        self.norm2 = layers.LayerNormalization()

        self.dropout1 = layers.Dropout(dropout)
        self.dropout2 = layers.Dropout(dropout)

    def call(self, inputs, training=None):
        attention_output = self.attention(
            inputs,
            inputs,
            training=training
        )

        x = self.norm1(
            inputs
            + self.dropout1(
                attention_output,
                training=training
            )
        )

        ffn_output = self.dense2(
            self.dense1(x)
        )

        return self.norm2(
            x
            + self.dropout2(
                ffn_output,
                training=training
            )
        )


## 9. Shared Patch Transformer 구성

In [ ]:
EMBED_DIM = 64
NUM_HEADS = 4
FF_DIM = 128
DROPOUT = 0.1

inputs = keras.Input(
    shape=(N_PATCHES, PATCH_LEN)
)

projection_layer = layers.Dense(EMBED_DIM)
x = projection_layer(inputs)

position_layer = LearnedPositionalEmbedding(
    N_PATCHES,
    EMBED_DIM
)
x = position_layer(x)

encoder1 = TransformerEncoder(
    EMBED_DIM,
    NUM_HEADS,
    FF_DIM,
    DROPOUT
)
x = encoder1(x)

encoder2 = TransformerEncoder(
    EMBED_DIM,
    NUM_HEADS,
    FF_DIM,
    DROPOUT
)
x = encoder2(x)

flatten_layer = layers.Flatten()
x = flatten_layer(x)

output_layer = layers.Dense(PRED_LEN)
outputs = output_layer(x)

model = keras.Model(
    inputs,
    outputs,
    name='channel_independent_patch_transformer_revin'
)

model.compile(
    optimizer=keras.optimizers.Adam(
        learning_rate=1e-4
    ),
    loss='mse',
    metrics=['mae']
)

model.summary()


## 10. Model Training

In [ ]:
early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

history = model.fit(
    X_train_ci,
    y_train_ci,
    validation_data=(
        X_val_ci,
        y_val_ci
    ),
    epochs=100,
    batch_size=128,
    shuffle=False,
    callbacks=[early_stopping],
    verbose=1
)


In [ ]:
plt.figure(figsize=(7, 4))

plt.plot(
    history.history['loss'],
    label='Train'
)
plt.plot(
    history.history['val_loss'],
    label='Validation'
)

plt.xlabel('Epoch')
plt.ylabel('RevIN-Normalized Residual MSE')
plt.title('Learning Curve')
plt.legend()
plt.grid(True)
plt.show()


## 11. RevIN⁻¹로 Forecast 복원

모델 출력은 normalized scale의 residual입니다.

저장해 둔 window mean/std를 사용하여 원래 scale로 다시 복원합니다.

In [ ]:
pred_res_ci = model.predict(
    X_test_ci,
    batch_size=512,
    verbose=1
)

B_test = X_test.shape[0]

# [B*C, H] -> [B, C, H] -> [B, H, C]
pred_res = pred_res_ci.reshape(
    B_test,
    NUM_CHANNELS,
    PRED_LEN
)

pred_res = np.transpose(
    pred_res,
    (0, 2, 1)
)

# Last observed value in the RevIN-normalized space.
baseline_norm = X_test[:, -1, :][:, None, :]

# Normalized absolute prediction.
y_pred_norm = baseline_norm + pred_res

# RevIN inverse transform.
y_pred_raw = (
    y_pred_norm * test_std
    + test_mean
)

print("Prediction shape:", y_pred_raw.shape)


## 12. OT-Only Evaluation

In [ ]:
OT_IDX = CHANNELS.index('OT')

y_test_ot_raw = y_test_raw[:, :, OT_IDX]
y_pred_ot_raw = y_pred_raw[:, :, OT_IDX]

# Original-scale metrics
ot_real_mse = mean_squared_error(
    y_test_ot_raw.reshape(-1),
    y_pred_ot_raw.reshape(-1)
)

ot_real_mae = mean_absolute_error(
    y_test_ot_raw.reshape(-1),
    y_pred_ot_raw.reshape(-1)
)

# Global train-set scale used only for comparable reporting.
ot_mean_global = report_scaler.mean_[OT_IDX]
ot_scale_global = report_scaler.scale_[OT_IDX]

y_test_ot_report = (
    y_test_ot_raw - ot_mean_global
) / ot_scale_global

y_pred_ot_report = (
    y_pred_ot_raw - ot_mean_global
) / ot_scale_global

ot_norm_mse = mean_squared_error(
    y_test_ot_report.reshape(-1),
    y_pred_ot_report.reshape(-1)
)

ot_norm_mae = mean_absolute_error(
    y_test_ot_report.reshape(-1),
    y_pred_ot_report.reshape(-1)
)

print('FS3: RevIN + Multivariate Shared Training / OT Evaluation')
print(f'Normalized MSE : {ot_norm_mse:.4f}')
print(f'Normalized MAE : {ot_norm_mae:.4f}')
print(f'MSE (°C²)      : {ot_real_mse:.4f}')
print(f'MAE (°C)       : {ot_real_mae:.4f}')


## 13. FS1 / FS2 / FS3 비교

In [ ]:
fs1 = {
    'Model': 'FS1: Univariate Patch',
    'Norm MSE': 0.0507,
    'Norm MAE': 0.1654,
    'MSE (°C²)': 3.5340,
    'MAE (°C)': 1.3808
}

fs2 = {
    'Model': 'FS2: Multivariate Shared',
    'Norm MSE': 0.0543,
    'Norm MAE': 0.1627,
    'MSE (°C²)': 3.7878,
    'MAE (°C)': 1.3584
}

fs3 = {
    'Model': 'FS3: Multivariate Shared + RevIN',
    'Norm MSE': ot_norm_mse,
    'Norm MAE': ot_norm_mae,
    'MSE (°C²)': ot_real_mse,
    'MAE (°C)': ot_real_mae
}

results = pd.DataFrame(
    [fs1, fs2, fs3]
)

display(results.round(4))


## 14. OT Forecast Example

In [ ]:
sample_idx = 0

past_ot_raw = X_test_raw[
    sample_idx,
    :,
    OT_IDX
]

future_ot_raw = y_test_ot_raw[
    sample_idx
]

pred_ot_raw = y_pred_ot_raw[
    sample_idx
]

past_x = np.arange(
    -LOOKBACK + 1,
    1
)

future_x = np.arange(
    1,
    PRED_LEN + 1
)

plt.figure(figsize=(10, 4))

plt.plot(
    past_x,
    past_ot_raw,
    label='Past OT'
)

plt.plot(
    future_x,
    future_ot_raw,
    label='Ground Truth'
)

plt.plot(
    future_x,
    pred_ot_raw,
    label='Patch Transformer + RevIN'
)

plt.axvline(
    0,
    linestyle='--'
)

plt.xlabel('Time Step')
plt.ylabel('OT (°C)')
plt.title('OT Forecast with RevIN')
plt.legend()
plt.grid(True)
plt.show()


## 15. Discussion

다음 질문을 생각해 보세요.

1. RevIN을 추가했을 때 OT 성능은 좋아졌나요?
2. Global standardization과 window-level normalization은 무엇이 다른가요?
3. Channel-independent 구조의 장점과 한계는 무엇인가요?

## 핵심 정리

```text
FS1: Univariate Patch Transformer
FS2: Multivariate + Channel Independent
FS3: Multivariate + Channel Independent + RevIN
```

세 실험을 비교하면서
**patching, channel 처리 방식, normalization**이 시계열 Transformer에 어떤 영향을 주는지 확인합니다.